# Response Clarity Classification with D3-Agentic Prompting

**Student ID:** sdi2200160  
**Course:** Artificial Intelligence II - Deep Learning for NLP  
**Assignment:** Homework 4 - D3-Agentic Prompting

This notebook is self-contained for Kaggle. It loads the CLARITY/QEvasion dataset, evaluates the required four-agent D3 pipeline with `Qwen/Qwen3.5-0.8B`, compares it with a single-agent Qwen comparator and previous-assignment baselines, selects by validation **macro F1**, and writes:

```text
submission_best_d3_agentic_system.csv
```

Macro F1 is the primary selection criterion because this task is sensitive to class collapse, especially on `Ambivalent`.

## Runtime Checks

Keep Internet enabled for the first Kaggle run. The first cell upgrades the Hugging Face runtime stack on Kaggle and installs a current Transformers source build when needed for the `qwen3_5` architecture. PyTorch is not upgraded inside Kaggle, because changing Kaggle's CUDA build can break GPU support.

In [ ]:
from __future__ import annotations

import importlib
import importlib.util
import os
from pathlib import Path
import platform
import subprocess
import sys

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')

IS_KAGGLE = bool(os.environ.get('KAGGLE_KERNEL_RUN_TYPE')) or Path('/kaggle').exists()
FORCE_MODERN_HF_RUNTIME = IS_KAGGLE
FORCE_TRANSFORMERS_SOURCE = IS_KAGGLE
REQUIRED_RUNTIME = {
    'datasets': 'datasets>=4.0.0',
    'accelerate': 'accelerate>=1.0.0',
    'sentencepiece': 'sentencepiece>=0.2.0',
    'huggingface_hub': 'huggingface_hub>=0.30.0',
    'safetensors': 'safetensors>=0.4.5',
    'tiktoken': 'tiktoken>=0.7.0',
}
TRANSFORMERS_PYPI_SPEC = 'transformers>=4.52.0'
TRANSFORMERS_SOURCE_SPECS = (
    'git+https://github.com/huggingface/transformers.git',
    'https://github.com/huggingface/transformers/archive/refs/heads/main.zip',
)
QWEN_MODEL_TYPE = 'qwen3_5'
QWEN_CONFIG_PROBE_MODEL = 'Qwen/Qwen3.5-0.8B'
CORE_MODULES = ('numpy', 'pandas', 'sklearn', 'torch')


def module_exists(module_name: str) -> bool:
    return importlib.util.find_spec(module_name) is not None


def purge_imported_package(package_name: str) -> None:
    for module_name in list(sys.modules):
        if module_name == package_name or module_name.startswith(package_name + '.'):
            del sys.modules[module_name]


def pip_install(pip_spec: str) -> None:
    print(f'[install] {pip_spec}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', pip_spec])
    importlib.invalidate_caches()


def ensure_package(module_name: str, pip_spec: str, force_upgrade: bool = False) -> None:
    if module_exists(module_name) and not force_upgrade:
        print(f'[ok] {module_name} already available')
        return
    try:
        pip_install(pip_spec)
    except subprocess.CalledProcessError as error:
        raise RuntimeError(f'Failed to install {pip_spec}. Enable Internet on Kaggle and rerun.') from error
    purge_imported_package(module_name)
    if not module_exists(module_name):
        raise RuntimeError(f'Installed {pip_spec}, but {module_name} is still unavailable.')


def transformers_supports_qwen35() -> bool:
    if not module_exists('transformers'):
        return False
    try:
        import transformers as transformers_probe
        from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    except Exception as error:
        print(f'[needs-install] could not inspect transformers: {error}')
        return False
    if QWEN_MODEL_TYPE in CONFIG_MAPPING_NAMES:
        print(f'[ok] transformers {transformers_probe.__version__} recognizes {QWEN_MODEL_TYPE}')
        return True
    try:
        transformers_probe.AutoConfig.from_pretrained(QWEN_CONFIG_PROBE_MODEL, trust_remote_code=True)
    except Exception as error:
        print(f'[needs-install] transformers {transformers_probe.__version__} cannot load {QWEN_CONFIG_PROBE_MODEL}: {error}')
        return False
    print(f'[ok] transformers {transformers_probe.__version__} loads {QWEN_CONFIG_PROBE_MODEL} through remote code')
    return True


def install_transformers_from_source() -> bool:
    for source_spec in TRANSFORMERS_SOURCE_SPECS:
        try:
            pip_install(source_spec)
        except subprocess.CalledProcessError as error:
            print(f'[warn] failed to install {source_spec}: {error}')
            continue
        purge_imported_package('transformers')
        if transformers_supports_qwen35():
            return True
    return False


def ensure_transformers_for_qwen35() -> None:
    if FORCE_TRANSFORMERS_SOURCE:
        print('[info] Kaggle runtime detected; installing Transformers from source for Qwen3.5 support')
        if install_transformers_from_source():
            return
    ensure_package('transformers', TRANSFORMERS_PYPI_SPEC, force_upgrade=FORCE_MODERN_HF_RUNTIME)
    if transformers_supports_qwen35():
        return
    if install_transformers_from_source():
        return
    raise RuntimeError('Transformers still does not recognize qwen3_5. Enable Internet and rerun.')


for module_name, pip_spec in REQUIRED_RUNTIME.items():
    ensure_package(module_name, pip_spec, force_upgrade=FORCE_MODERN_HF_RUNTIME)
ensure_transformers_for_qwen35()

missing_core = [module_name for module_name in CORE_MODULES if not module_exists(module_name)]
if missing_core:
    raise ImportError('Missing core dependencies: ' + ', '.join(missing_core))

import datasets
import torch
import transformers

print(f'Python:       {sys.version.split()[0]}')
print(f'Interpreter:  {sys.executable}')
print(f'Platform:     {platform.platform()}')
print(f'Kaggle:       {IS_KAGGLE}')
print(f'torch:        {torch.__version__} | CUDA available: {torch.cuda.is_available()}')
print(f'transformers: {transformers.__version__}')
print(f'datasets:     {datasets.__version__}')

if IS_KAGGLE and not torch.cuda.is_available():
    print('Warning: Kaggle GPU is not enabled. Qwen inference will be very slow.')

In [ ]:
import dataclasses
import gc
import json
import random
import re
import typing
from collections.abc import Iterable

import numpy as np
import pandas as pd
import sklearn.metrics
import sklearn.model_selection
from IPython.display import display

## Configuration

The required core experiment uses the four D3 agents with `Qwen/Qwen3.5-0.8B`. A single-agent comparator using the same model and label guidance is included to isolate the effect of decomposition without changing model scale. No hyperparameter sweep is performed; the goal is a fixed, reproducible agentic design.

In [ ]:
RANDOM_STATE = 42
OUTPUT_DIR = Path('runs_hw4_kaggle')
SUBMISSION_FILENAME = 'submission_best_d3_agentic_system.csv'

MODELS = {
    'qwen-0.8b': 'Qwen/Qwen3.5-0.8B',
}
MODEL_KEY = 'qwen-0.8b'
MODEL_NAME = MODELS[MODEL_KEY]
EXPERIMENTS_TO_RUN = ['d3-agentic', 'single-agent']

VALIDATION_FRACTION = 0.20
VALIDATION_PER_LABEL = 10
MAX_QUESTION_CHARS = 300
MAX_ANSWER_CHARS = 900
MAX_INTERMEDIATE_CHARS = 1200
MAX_NEW_TOKENS = 96
DECISION_MAX_NEW_TOKENS = 48
BATCH_SIZE = 1
DEVICE_MAP = 'auto'
TORCH_DTYPE = 'auto'
USE_CHAT_TEMPLATE = True
ENABLE_THINKING = False
SYSTEM_MESSAGE = 'You are a careful coordinator for response-clarity annotation agents.'

CLARITY_LABELS = ('Clear Reply', 'Ambivalent', 'Clear Non-Reply')
SELECTION_METRIC = 'f1_macro'
SELECTION_TIE_BREAKER = 'accuracy'

## Data Loading

In [ ]:
def seed_everything(seed: int = RANDOM_STATE) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('HF_TOKEN')
        if token:
            os.environ['HF_TOKEN'] = token
    except Exception:
        pass
else:
    try:
        import dotenv
        dotenv.load_dotenv(override=True)
    except Exception:
        pass


def clean_text(value: typing.Any) -> str:
    if pd.isna(value):
        return ''
    return ' '.join(str(value).split())


def canonical_clarity_label(value: typing.Any) -> str:
    text = clean_text(value)
    normalized = re.sub(r'[-_]+', ' ', text).casefold()
    normalized = re.sub(r'\s+', ' ', normalized).strip()
    if normalized in {'clear reply', 'explicit'}:
        return 'Clear Reply'
    if normalized in {'ambivalent', 'ambivalent reply'}:
        return 'Ambivalent'
    if normalized in {'clear non reply', 'clear not reply', 'clear non-reply', 'clear not-reply'}:
        return 'Clear Non-Reply'
    return text


def canonical_evasion_label(value: typing.Any) -> str:
    text = clean_text(value)
    text = re.sub(r'^\d+(?:\.\d+)*\s*', '', text)
    aliases = {'Declining': 'Declining to answer', 'Ignorance': 'Claims ignorance', 'Partial': 'Partial/half-answer'}
    return aliases.get(text, text)


def normalize_frame(frame: pd.DataFrame, require_label: bool) -> pd.DataFrame:
    required = ['question', 'interview_answer']
    missing = [column for column in required if column not in frame]
    if missing:
        raise ValueError(f'Missing required columns: {missing}')
    if require_label and 'clarity_label' not in frame:
        raise ValueError('Training data must include clarity_label.')
    result = frame.copy()
    if 'clarity_label' not in result:
        result['clarity_label'] = ''
    if 'evasion_label' not in result:
        result['evasion_label'] = ''
    result['question'] = result['question'].map(clean_text)
    result['interview_answer'] = result['interview_answer'].map(clean_text)
    result['clarity_label'] = result['clarity_label'].map(canonical_clarity_label)
    result['evasion_label'] = result['evasion_label'].map(canonical_evasion_label)
    return result[['question', 'interview_answer', 'clarity_label', 'evasion_label']].fillna('')


def balanced_label_sample(frame: pd.DataFrame, per_label: int, seed: int = RANDOM_STATE) -> pd.DataFrame:
    if per_label <= 0:
        return frame.reset_index(drop=True)
    parts = []
    for i, label in enumerate(CLARITY_LABELS):
        group = frame[frame['clarity_label'] == label]
        if len(group) < per_label:
            raise ValueError(f'Cannot sample {per_label} examples for {label}; only {len(group)} available.')
        parts.append(group.sample(n=per_label, random_state=seed + i))
    return pd.concat(parts).sort_index().reset_index(drop=True)


seed_everything(RANDOM_STATE)
raw_data = datasets.load_dataset('ailsntua/QEvasion')
train_all = normalize_frame(raw_data['train'].to_pandas(), require_label=True)
test_df = normalize_frame(raw_data['test'].to_pandas(), require_label=False)
train_fit, validation_df = sklearn.model_selection.train_test_split(
    train_all,
    test_size=VALIDATION_FRACTION,
    random_state=RANDOM_STATE,
    stratify=train_all['clarity_label'],
)
train_fit = train_fit.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
evaluation_df = balanced_label_sample(validation_df, VALIDATION_PER_LABEL, seed=RANDOM_STATE)

print(f'Train for comparison: {len(train_fit)}')
print(f'Validation split:     {len(validation_df)}')
print(f'Validation sample:    {len(evaluation_df)}')
print(f'Test split:           {len(test_df)}')
display(evaluation_df['clarity_label'].value_counts().rename_axis('label').reset_index(name='count'))

## Previous-Assignment Baselines

In [ ]:
previous_baselines = pd.DataFrame([
    {'homework': 'HW1', 'system': 'TF-IDF + Logistic Regression', 'evaluation': 'official test', 'accuracy': 0.63, 'f1_macro': 0.45, 'note': 'Best classical baseline; approximate values from HW1 report.'},
    {'homework': 'HW1', 'system': 'GloVe Wiki + Logistic Regression', 'evaluation': 'official test', 'accuracy': np.nan, 'f1_macro': 0.39, 'note': 'Mean-pooled embedding baseline.'},
    {'homework': 'HW1', 'system': 'GloVe Twitter + Logistic Regression', 'evaluation': 'official test', 'accuracy': np.nan, 'f1_macro': 0.38, 'note': 'Mean-pooled conversational embedding baseline.'},
    {'homework': 'HW2', 'system': 'BERT-base fine-tuning', 'evaluation': 'official test', 'accuracy': 0.6299, 'f1_macro': 0.5561, 'note': 'Best encoder-only transformer.'},
    {'homework': 'HW2', 'system': 'DistilBERT fine-tuning', 'evaluation': 'official test', 'accuracy': 0.5779, 'f1_macro': 0.5066, 'note': 'Smaller encoder-only transformer.'},
    {'homework': 'HW2', 'system': 'DeBERTa-v3 fine-tuning', 'evaluation': 'official test', 'accuracy': 0.6688, 'f1_macro': 0.2672, 'note': 'Unstable run; collapsed to all Ambivalent predictions.'},
    {'homework': 'HW3', 'system': 'Qwen 4B few-shot prompting', 'evaluation': 'balanced validation', 'accuracy': 0.4667, 'f1_macro': 0.4667, 'note': 'Best single-invocation prompting run by macro F1.'},
    {'homework': 'HW3', 'system': 'Qwen 0.8B few-shot prompting', 'evaluation': 'balanced validation', 'accuracy': 0.4667, 'f1_macro': 0.4091, 'note': 'Same model scale as the required D3 system.'},
])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
previous_baselines.to_csv(OUTPUT_DIR / 'previous_assignment_baselines.csv', index=False)
display(previous_baselines)

## Parsing and Generation

In [ ]:
@dataclasses.dataclass(frozen=True)
class ParsedGeneration:
    raw_text: str
    label: str | None
    valid: bool
    parse_method: str
    rationale: str | None = None
    message: str = ''


class LabelNormalizer:
    def __init__(self) -> None:
        aliases = {
            'Clear Reply': ['clear reply', 'direct reply', 'clear answer', 'direct answer', 'explicit'],
            'Ambivalent': ['ambivalent', 'ambivalent reply', 'partial reply', 'ambiguous', 'implicit', 'dodging', 'general', 'deflection', 'partial', 'partial half answer'],
            'Clear Non-Reply': ['clear non reply', 'clear not reply', 'clear non-reply', 'non reply', 'non-reply', 'not reply', 'declining', 'ignorance', 'clarification'],
        }
        self.alias_to_label = {}
        for label, values in aliases.items():
            for value in [label, *values]:
                self.alias_to_label[self._key(value)] = label

    def normalize(self, value: typing.Any) -> str | None:
        text = self._strip_noise(self._key(str(value)))
        return self.alias_to_label.get(text)

    def find_mentions(self, text: str) -> list[tuple[int, str]]:
        normalized = self._key(text)
        mentions = []
        for alias, label in self.alias_to_label.items():
            pattern = r'(?<!\w)' + re.escape(alias).replace(r'\ ', r'[\s-]+') + r'(?!\w)'
            for match in re.finditer(pattern, normalized):
                mentions.append((match.start(), label))
        return sorted(mentions)

    @staticmethod
    def _key(text: str) -> str:
        text = text.casefold()
        text = re.sub(r'[`"\'*_{}\[\]().,:;!?]', ' ', text)
        text = re.sub(r'[/_-]+', ' ', text)
        return re.sub(r'\s+', ' ', text).strip()

    @staticmethod
    def _strip_noise(text: str) -> str:
        text = re.sub(r'^(the\s+)?(final\s+)?(label|answer|prediction|class|category)\s+(is|=|:)?\s+', '', text)
        text = re.sub(r'\s+(because|since|as)\s+.*$', '', text)
        return text.strip()


class GenerationParser:
    def __init__(self, default_label: str | None = None) -> None:
        self.normalizer = LabelNormalizer()
        self.default_label = default_label

    def parse(self, text: typing.Any) -> ParsedGeneration:
        raw = '' if text is None else str(text).strip()
        for obj in reversed(list(self._iter_json_objects(raw))):
            if isinstance(obj, dict):
                for key in ('label', 'final_label', 'predicted_label', 'prediction', 'class', 'category'):
                    if key in obj:
                        label = self.normalizer.normalize(obj[key])
                        if label:
                            rationale = None
                            for rkey in ('rationale', 'reason', 'explanation', 'analysis'):
                                if rkey in obj and obj[rkey] is not None:
                                    rationale = str(obj[rkey]).strip()
                                    break
                            return ParsedGeneration(raw, label, True, 'json', rationale)
        for match in reversed(list(re.finditer(r'(?im)^\s*(?:final\s+)?(?:label|answer|prediction|class|category)\s*[:=-]\s*(.+?)\s*$', raw))):
            label = self.normalizer.normalize(match.group(1))
            if label:
                return ParsedGeneration(raw, label, True, 'labeled_line')
        mentions = self.normalizer.find_mentions(raw)
        if mentions:
            unique = {label for _, label in mentions}
            if len(unique) == 1:
                return ParsedGeneration(raw, next(iter(unique)), True, 'single_mention')
            last_pos, last_label = mentions[-1]
            window = self.normalizer._key(raw)[max(0, last_pos - 40): last_pos + 120]
            if re.search(r'\b(final|therefore|answer|label|prediction)\b', window):
                return ParsedGeneration(raw, last_label, True, 'final_mention')
            return ParsedGeneration(raw, None, False, 'ambiguous_mentions', message=f'Multiple labels mentioned: {sorted(unique)}')
        if self.default_label:
            return ParsedGeneration(raw, self.default_label, False, 'default', message='Default label used.')
        return ParsedGeneration(raw, None, False, 'invalid', message='No valid label found.')

    def parse_many(self, texts: Iterable[typing.Any]) -> list[ParsedGeneration]:
        return [self.parse(text) for text in texts]

    @staticmethod
    def _iter_json_objects(text: str):
        decoder = json.JSONDecoder()
        for start in [match.start() for match in re.finditer(r'\{', text)]:
            try:
                parsed, _ = decoder.raw_decode(text[start:])
            except json.JSONDecodeError:
                continue
            yield parsed


@dataclasses.dataclass(frozen=True)
class DecodingConfig:
    max_new_tokens: int = MAX_NEW_TOKENS
    do_sample: bool = False
    temperature: float = 0.0
    top_p: float = 1.0
    repetition_penalty: float = 1.0


class HuggingFaceGenerator:
    def __init__(self, model_name: str, decoding: DecodingConfig, batch_size: int = BATCH_SIZE) -> None:
        self.model_name = model_name
        self.decoding = decoding
        self.batch_size = batch_size
        self._tokenizer = None
        self._processor = None
        self._model = None
        self._loaded_backend = None

    def generate(self, prompts: list[str]) -> list[str]:
        self._load()
        if self._loaded_backend == 'image-text-to-text':
            return self._generate_image_text_to_text(prompts)
        return self._generate_causal_lm(prompts)

    def _load(self) -> None:
        if self._model is not None:
            return
        if hasattr(transformers, 'AutoModelForImageTextToText'):
            try:
                self._processor = transformers.AutoProcessor.from_pretrained(self.model_name, trust_remote_code=True)
                self._ensure_processor_padding_token()
                model_cls = getattr(transformers, 'AutoModelForImageTextToText')
                self._model = model_cls.from_pretrained(self.model_name, device_map=DEVICE_MAP, torch_dtype=TORCH_DTYPE, trust_remote_code=True)
                self._sync_generation_config_token_ids()
                self._model.eval()
                self._loaded_backend = 'image-text-to-text'
                return
            except Exception as error:
                print(f'Image-text-to-text load failed for {self.model_name}: {error}')
                self._processor = None
                self._model = None
        self._tokenizer = transformers.AutoTokenizer.from_pretrained(self.model_name, trust_remote_code=True)
        if self._tokenizer.pad_token_id is None:
            self._tokenizer.pad_token = self._tokenizer.eos_token
        self._model = transformers.AutoModelForCausalLM.from_pretrained(self.model_name, device_map=DEVICE_MAP, torch_dtype=TORCH_DTYPE, trust_remote_code=True)
        self._model.eval()
        self._loaded_backend = 'causal-lm'

    def _generate_causal_lm(self, prompts: list[str]) -> list[str]:
        outputs = []
        for start in range(0, len(prompts), self.batch_size):
            batch_prompts = [self._format_prompt(prompt) for prompt in prompts[start:start + self.batch_size]]
            encoded = self._tokenizer(batch_prompts, return_tensors='pt', padding=True)
            encoded = self._move_batch(encoded)
            input_width = encoded['input_ids'].shape[1]
            with torch.no_grad():
                generated = self._model.generate(**encoded, pad_token_id=self._tokenizer.pad_token_id, eos_token_id=self._tokenizer.eos_token_id, **self._generation_kwargs())
            for sequence in generated:
                outputs.append(self._tokenizer.decode(sequence[input_width:], skip_special_tokens=True).strip())
        return outputs

    def _generate_image_text_to_text(self, prompts: list[str]) -> list[str]:
        outputs = []
        for start in range(0, len(prompts), self.batch_size):
            messages = [self._messages(prompt, structured_content=True) for prompt in prompts[start:start + self.batch_size]]
            batch_text = [self._apply_chat_template(self._processor, message, tokenize=False, add_generation_prompt=True) for message in messages]
            encoded = self._processor(text=batch_text, return_tensors='pt', padding=True)
            encoded = self._move_batch(encoded)
            input_width = encoded['input_ids'].shape[1]
            with torch.no_grad():
                generated = self._model.generate(**encoded, **self._generation_token_kwargs(), **self._generation_kwargs())
            outputs.extend(text.strip() for text in self._processor.batch_decode(generated[:, input_width:], skip_special_tokens=True))
        return outputs

    def _format_prompt(self, prompt: str) -> str:
        if not USE_CHAT_TEMPLATE or self._tokenizer is None or not getattr(self._tokenizer, 'chat_template', None):
            return prompt
        return self._apply_chat_template(self._tokenizer, self._messages(prompt, structured_content=False), tokenize=False, add_generation_prompt=True)

    def _messages(self, prompt: str, structured_content: bool) -> list[dict[str, typing.Any]]:
        messages = []
        content = [{'type': 'text', 'text': SYSTEM_MESSAGE}] if structured_content else SYSTEM_MESSAGE
        messages.append({'role': 'system', 'content': content})
        content = [{'type': 'text', 'text': prompt}] if structured_content else prompt
        messages.append({'role': 'user', 'content': content})
        return messages

    def _apply_chat_template(self, owner, messages, **kwargs):
        if ENABLE_THINKING is not None:
            kwargs['enable_thinking'] = ENABLE_THINKING
        try:
            return owner.apply_chat_template(messages, **kwargs)
        except TypeError as error:
            if 'enable_thinking' not in kwargs:
                raise
            if 'enable_thinking' not in str(error) and 'unexpected keyword' not in str(error):
                raise
            kwargs.pop('enable_thinking')
            return owner.apply_chat_template(messages, **kwargs)

    def _generation_kwargs(self) -> dict[str, typing.Any]:
        kwargs = {'max_new_tokens': self.decoding.max_new_tokens, 'do_sample': self.decoding.do_sample, 'repetition_penalty': self.decoding.repetition_penalty}
        if self.decoding.do_sample:
            kwargs.update({'temperature': self.decoding.temperature, 'top_p': self.decoding.top_p})
        return kwargs

    def _generation_token_kwargs(self) -> dict[str, typing.Any]:
        eos = self._token_id('eos_token_id')
        pad = self._token_id('pad_token_id') or eos
        kwargs = {}
        if pad is not None:
            kwargs['pad_token_id'] = pad
        if eos is not None:
            kwargs['eos_token_id'] = eos
        return kwargs

    def _processor_tokenizer(self):
        return getattr(self._processor, 'tokenizer', None) if self._processor is not None else None

    def _ensure_processor_padding_token(self) -> None:
        tokenizer = self._processor_tokenizer()
        if tokenizer is not None and getattr(tokenizer, 'pad_token_id', None) is None and getattr(tokenizer, 'eos_token', None) is not None:
            tokenizer.pad_token = tokenizer.eos_token

    def _sync_generation_config_token_ids(self) -> None:
        generation_config = getattr(self._model, 'generation_config', None)
        if generation_config is None:
            return
        eos = self._token_id('eos_token_id')
        pad = self._token_id('pad_token_id') or eos
        if getattr(generation_config, 'pad_token_id', None) is None and pad is not None:
            generation_config.pad_token_id = pad
        if getattr(generation_config, 'eos_token_id', None) is None and eos is not None:
            generation_config.eos_token_id = eos

    def _token_id(self, name: str):
        tokenizer = self._tokenizer or self._processor_tokenizer()
        value = getattr(tokenizer, name, None) if tokenizer is not None else None
        if value is not None:
            return value
        generation_config = getattr(self._model, 'generation_config', None) if self._model is not None else None
        return getattr(generation_config, name, None)

    def _move_batch(self, batch):
        device = getattr(self._model, 'device', None)
        if hasattr(batch, 'to') and device is not None:
            return batch.to(device)
        if device is None:
            return batch
        return {key: value.to(device) if hasattr(value, 'to') else value for key, value in batch.items()}

## D3-Agentic Prompting

In [ ]:
EVASION_SUBTYPES = {
    'Clear Reply': ['Explicit'],
    'Ambivalent': ['Implicit', 'Dodging', 'General', 'Deflection', 'Partial/half-answer'],
    'Clear Non-Reply': ['Declining to answer', 'Claims ignorance', 'Clarification'],
}


def truncate_text(text: str, max_chars: int | None) -> str:
    text = clean_text(text)
    if max_chars is None or len(text) <= max_chars:
        return text
    marker = ' ... '
    available = max_chars - len(marker)
    if available <= 0:
        return text[:max_chars]
    head = max(1, available // 2)
    tail = max(1, available - head)
    return text[:head].rstrip() + marker + text[-tail:].lstrip()


class D3PromptBuilder:
    def taxonomy_block(self) -> str:
        lines = ['Dataset label taxonomy:']
        for label in CLARITY_LABELS:
            lines.append(f'- {label}: {", ".join(EVASION_SUBTYPES[label])}')
        return '\n'.join(lines)

    def question(self, record) -> str:
        return truncate_text(record.get('question', ''), MAX_QUESTION_CHARS)

    def answer(self, record) -> str:
        return truncate_text(record.get('interview_answer', ''), MAX_ANSWER_CHARS)

    def intermediate(self, value: str) -> str:
        return truncate_text(value, MAX_INTERMEDIATE_CHARS)

    def question_intent_prompt(self, record) -> str:
        return '\n\n'.join([
            'You are the Question Intent Agent in a D3 response-clarity pipeline.',
            'Identify what the journalist is asking for. Do not classify the answer.',
            f'Question:\n{self.question(record)}',
            'Return valid JSON only with keys: "asked_for", "required_information", "question_focus". Keep values brief.',
        ])

    def answer_content_prompt(self, record) -> str:
        return '\n\n'.join([
            'You are the Answer Content Agent in a D3 response-clarity pipeline.',
            'Extract what the answer actually says. Do not classify the answer.',
            f'Question:\n{self.question(record)}',
            f'Answer:\n{self.answer(record)}',
            'Return valid JSON only with keys: "explicit_claims", "relevant_details", "hedges_or_qualifiers", "omitted_points".',
        ])

    def gap_evasion_prompt(self, record, intent: str, content: str) -> str:
        return '\n\n'.join([
            'You are the Gap and Evasion Agent in a D3 response-clarity pipeline.',
            'Compare expected answer content with actual answer content. Do not output the final clarity label.',
            self.taxonomy_block(),
            f'Question:\n{self.question(record)}',
            f'Answer:\n{self.answer(record)}',
            'Question Intent Agent output:\n' + self.intermediate(intent),
            'Answer Content Agent output:\n' + self.intermediate(content),
            'Return valid JSON only with keys: "matched_requirements", "missing_requirements", "evasion_patterns", "responsiveness_summary", "likely_subtype".',
        ])

    def decision_prompt(self, record, intent: str, content: str, gap: str) -> str:
        labels = ' | '.join(CLARITY_LABELS)
        return '\n\n'.join([
            'You are the Decision Agent in a D3 response-clarity pipeline.',
            'Assign exactly one final clarity label using the previous agents.',
            'Decision rules:',
            '- Clear Reply: explicitly and specifically answers the main question.',
            '- Ambivalent: partial, implicit, generic, hedged, deflective, or only partly responsive.',
            '- Clear Non-Reply: refuses, redirects, asks for clarification, claims ignorance, or gives no substantive answer.',
            '- Judge relative to the exact question; do not reward length or topical fluency.',
            self.taxonomy_block(),
            f'Question:\n{self.question(record)}',
            f'Answer:\n{self.answer(record)}',
            'Question Intent Agent output:\n' + self.intermediate(intent),
            'Answer Content Agent output:\n' + self.intermediate(content),
            'Gap and Evasion Agent output:\n' + self.intermediate(gap),
            f'Return valid JSON only: {{"label": "<{labels}>", "rationale": "<brief evidence-based reason>"}}.',
        ])

    def direct_prompt(self, record) -> str:
        labels = ' | '.join(CLARITY_LABELS)
        return '\n\n'.join([
            "Classify the interview answer's responsiveness to the question.",
            f'Use exactly one label: {labels}.',
            self.taxonomy_block(),
            'Decision rules:',
            '- Clear Reply only if the answer explicitly and specifically answers the main question.',
            '- Ambivalent for partial, implicit, generic, hedged, deflective, or partly responsive answers.',
            '- Clear Non-Reply for refusals, redirections, clarification requests, ignorance claims, or no substantive answer.',
            f'Question:\n{self.question(record)}',
            f'Answer:\n{self.answer(record)}',
            f'Return valid JSON only: {{"label": "<{labels}>", "rationale": "<brief evidence-based reason>"}}.',
        ])


class D3AgenticClassifier:
    def __init__(self, generator: HuggingFaceGenerator, builder: D3PromptBuilder | None = None) -> None:
        self.generator = generator
        self.builder = builder or D3PromptBuilder()
        self.parser = GenerationParser()

    def generate_frame(self, source: pd.DataFrame, include_prompts: bool = True) -> pd.DataFrame:
        source = source.copy()
        intent_prompts = [self.builder.question_intent_prompt(record) for _, record in source.iterrows()]
        intent_generations = self.generator.generate(intent_prompts)
        content_prompts = [self.builder.answer_content_prompt(record) for _, record in source.iterrows()]
        content_generations = self.generator.generate(content_prompts)
        gap_prompts = [self.builder.gap_evasion_prompt(record, intent, content) for (_, record), intent, content in zip(source.iterrows(), intent_generations, content_generations)]
        gap_generations = self.generator.generate(gap_prompts)
        decision_prompts = [self.builder.decision_prompt(record, intent, content, gap) for (_, record), intent, content, gap in zip(source.iterrows(), intent_generations, content_generations, gap_generations)]
        decision_generations = self.generator.generate(decision_prompts)
        parsed = self.parser.parse_many(decision_generations)
        frame = pd.DataFrame({
            'question_intent_generation': intent_generations,
            'answer_content_generation': content_generations,
            'gap_evasion_generation': gap_generations,
            'decision_generation': decision_generations,
            'Predicted': [item.label for item in parsed],
            'valid': [item.valid for item in parsed],
            'parse_method': [item.parse_method for item in parsed],
            'parse_message': [item.message for item in parsed],
            'rationale': [item.rationale for item in parsed],
        }, index=source.index)
        if include_prompts:
            frame.insert(0, 'question_intent_prompt', intent_prompts)
            frame.insert(2, 'answer_content_prompt', content_prompts)
            frame.insert(4, 'gap_evasion_prompt', gap_prompts)
            frame.insert(6, 'decision_prompt', decision_prompts)
        return frame


class SingleAgentComparator:
    def __init__(self, generator: HuggingFaceGenerator, builder: D3PromptBuilder | None = None) -> None:
        self.generator = generator
        self.builder = builder or D3PromptBuilder()
        self.parser = GenerationParser()

    def generate_frame(self, source: pd.DataFrame, include_prompts: bool = True) -> pd.DataFrame:
        prompts = [self.builder.direct_prompt(record) for _, record in source.iterrows()]
        generations = self.generator.generate(prompts)
        parsed = self.parser.parse_many(generations)
        frame = pd.DataFrame({
            'decision_generation': generations,
            'Predicted': [item.label for item in parsed],
            'valid': [item.valid for item in parsed],
            'parse_method': [item.parse_method for item in parsed],
            'parse_message': [item.message for item in parsed],
            'rationale': [item.rationale for item in parsed],
        }, index=source.index)
        if include_prompts:
            frame.insert(0, 'decision_prompt', prompts)
        return frame


def make_classifier(experiment: str):
    if experiment == 'd3-agentic':
        return D3AgenticClassifier(HuggingFaceGenerator(MODEL_NAME, DecodingConfig(max_new_tokens=MAX_NEW_TOKENS)))
    if experiment == 'single-agent':
        return SingleAgentComparator(HuggingFaceGenerator(MODEL_NAME, DecodingConfig(max_new_tokens=DECISION_MAX_NEW_TOKENS)))
    raise ValueError(f'Unknown experiment: {experiment}')

## Evaluation Helpers

In [ ]:
def classification_scores(true: Iterable[str], pred: Iterable[str | None]) -> dict[str, float]:
    y_true = pd.Series(list(true), dtype='object')
    y_pred = pd.Series(list(pred), dtype='object')
    valid = y_pred.isin(CLARITY_LABELS)
    y_eval = y_pred.where(valid, '__invalid__')
    return {
        'accuracy': float(sklearn.metrics.accuracy_score(y_true, y_eval)),
        'precision_macro': float(sklearn.metrics.precision_score(y_true, y_eval, labels=list(CLARITY_LABELS), average='macro', zero_division=0)),
        'recall_macro': float(sklearn.metrics.recall_score(y_true, y_eval, labels=list(CLARITY_LABELS), average='macro', zero_division=0)),
        'f1_macro': float(sklearn.metrics.f1_score(y_true, y_eval, labels=list(CLARITY_LABELS), average='macro', zero_division=0)),
        'f1_weighted': float(sklearn.metrics.f1_score(y_true, y_eval, labels=list(CLARITY_LABELS), average='weighted', zero_division=0)),
        'f1_micro': float(sklearn.metrics.f1_score(y_true, y_eval, labels=list(CLARITY_LABELS), average='micro', zero_division=0)),
        'invalid_rate': float((~valid).mean()) if len(valid) else 0.0,
    }


def confusion_matrix_frame(true: Iterable[str], pred: Iterable[str | None]) -> pd.DataFrame:
    y_true = pd.Series(list(true), dtype='object')
    y_pred = pd.Series(list(pred), dtype='object')
    labels = list(CLARITY_LABELS)
    if (~y_pred.isin(CLARITY_LABELS)).any():
        y_pred = y_pred.where(y_pred.isin(CLARITY_LABELS), '__invalid__')
        labels.append('__invalid__')
    return pd.DataFrame(sklearn.metrics.confusion_matrix(y_true, y_pred, labels=labels), index=labels, columns=labels)


def classification_report_frame(true: Iterable[str], pred: Iterable[str | None]) -> pd.DataFrame:
    y_true = pd.Series(list(true), dtype='object')
    y_pred = pd.Series(list(pred), dtype='object')
    y_eval = y_pred.where(y_pred.isin(CLARITY_LABELS), '__invalid__')
    return pd.DataFrame(sklearn.metrics.classification_report(y_true, y_eval, labels=list(CLARITY_LABELS), output_dict=True, zero_division=0)).transpose()


def length_subgroup_scores(frame: pd.DataFrame) -> pd.DataFrame:
    work = frame.copy()
    work['question_words'] = work['question'].fillna('').astype(str).str.split().str.len()
    work['answer_words'] = work['interview_answer'].fillna('').astype(str).str.split().str.len()
    rows = []
    for base_col in ['question_words', 'answer_words']:
        try:
            work[f'{base_col}_bin'] = pd.qcut(work[base_col], q=3, duplicates='drop').astype(str)
        except ValueError:
            work[f'{base_col}_bin'] = 'all'
        for group, group_frame in work.groupby(f'{base_col}_bin', dropna=False):
            row = {'group_col': f'{base_col}_bin', 'group': group, 'n': len(group_frame)}
            row.update(classification_scores(group_frame['clarity_label'], group_frame['Predicted']))
            rows.append(row)
    return pd.DataFrame(rows)


def submission_frame(predictions: Iterable[str | None], invalid_label: str = 'Ambivalent') -> pd.DataFrame:
    values = [prediction if prediction in CLARITY_LABELS else invalid_label for prediction in predictions]
    frame = pd.DataFrame({'Predicted': values})
    frame.index.name = 'Id'
    return frame

## Validation Experiments

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'runs').mkdir(parents=True, exist_ok=True)
rows = []
run_frames = {}

for experiment in EXPERIMENTS_TO_RUN:
    print('=' * 80)
    print(f'MODEL:      {MODEL_KEY} -> {MODEL_NAME}')
    print(f'EXPERIMENT: {experiment}')
    classifier = make_classifier(experiment)
    frame = classifier.generate_frame(evaluation_df, include_prompts=True)
    joined = pd.concat([evaluation_df.reset_index(drop=True), frame.reset_index(drop=True)], axis=1)
    scores = classification_scores(joined['clarity_label'], joined['Predicted'])
    row = {'model_key': MODEL_KEY, 'model': MODEL_NAME, 'strategy': experiment, 'run_id': f'{MODEL_KEY}_{experiment}', 'n_validation': len(joined), 'calls_per_example': 4 if experiment == 'd3-agentic' else 1}
    row.update(scores)
    for label in CLARITY_LABELS:
        row[f'truth_{label}'] = int((joined['clarity_label'] == label).sum())
        row[f'pred_{label}'] = int((joined['Predicted'] == label).sum())
    row['valid_rate'] = float(joined['valid'].mean())
    rows.append(row)
    run_frames[experiment] = joined
    joined.to_csv(OUTPUT_DIR / 'runs' / f'{experiment}.generations.csv', index=False)
    classification_report_frame(joined['clarity_label'], joined['Predicted']).to_csv(OUTPUT_DIR / 'runs' / f'{experiment}.classification_report.csv')
    confusion_matrix_frame(joined['clarity_label'], joined['Predicted']).to_csv(OUTPUT_DIR / 'runs' / f'{experiment}.confusion.csv')
    length_subgroup_scores(joined).to_csv(OUTPUT_DIR / 'runs' / f'{experiment}.length_subgroups.csv', index=False)
    display(pd.DataFrame([row])[['strategy', 'f1_macro', 'accuracy', 'precision_macro', 'recall_macro', 'invalid_rate', 'valid_rate']].round(4))
    del classifier
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

results_df = pd.DataFrame(rows).sort_values([SELECTION_METRIC, SELECTION_TIE_BREAKER], ascending=[False, False], na_position='last').reset_index(drop=True)
results_df.to_csv(OUTPUT_DIR / 'experiment_summary.csv', index=False)
display(results_df[['model_key', 'strategy', 'f1_macro', 'accuracy', 'precision_macro', 'recall_macro', 'invalid_rate', 'calls_per_example']].round(4))

comparison_df = pd.concat([
    previous_baselines,
    results_df.assign(homework='HW4', system=results_df['model_key'] + ' ' + results_df['strategy'], evaluation='balanced validation', note='Current D3-agentic assignment run.')[['homework', 'system', 'evaluation', 'accuracy', 'f1_macro', 'note']]
], ignore_index=True)
comparison_df.to_csv(OUTPUT_DIR / 'baseline_comparison.csv', index=False)
display(comparison_df)

## Best System Selection and Error Analysis

In [ ]:
best_row = results_df.iloc[0].to_dict()
best_strategy = str(best_row['strategy'])
with open(OUTPUT_DIR / 'best_validation_system.json', 'w', encoding='utf-8') as file:
    json.dump({**best_row, 'selection_metric': SELECTION_METRIC, 'selection_tie_breaker': SELECTION_TIE_BREAKER}, file, indent=2)

print('Selected best system by validation macro F1:')
print(f'Model:    {MODEL_KEY} -> {MODEL_NAME}')
print(f'Strategy: {best_strategy}')
print(f'Macro F1: {best_row["f1_macro"]:.4f}')
print(f'Accuracy: {best_row["accuracy"]:.4f}')

best_frame = run_frames[best_strategy]
print('\nConfusion matrix:')
display(confusion_matrix_frame(best_frame['clarity_label'], best_frame['Predicted']))
print('\nClassification report:')
display(classification_report_frame(best_frame['clarity_label'], best_frame['Predicted']).round(4))
print('\nPrediction distribution:')
display(best_frame['Predicted'].value_counts(dropna=False).rename_axis('Predicted').reset_index(name='count'))

errors = best_frame[best_frame['clarity_label'] != best_frame['Predicted']].copy()
errors['confusion'] = errors['clarity_label'] + ' -> ' + errors['Predicted'].fillna('__invalid__')
errors.to_csv(OUTPUT_DIR / 'best_errors.csv', index=False)
print(f'Errors: {len(errors)} / {len(best_frame)}')
display(errors['confusion'].value_counts().rename_axis('confusion').reset_index(name='count'))
cols = ['clarity_label', 'Predicted', 'question', 'interview_answer', 'question_intent_generation', 'answer_content_generation', 'gap_evasion_generation', 'rationale']
display(errors[[c for c in cols if c in errors]].head(8))

## Final Kaggle Submission

In [ ]:
full_train = train_all.reset_index(drop=True)
print(f'Rerunning selected system on full test split: {best_strategy}')
final_classifier = make_classifier(best_strategy)
final_generations = final_classifier.generate_frame(test_df, include_prompts=True)
final_generations.to_csv(OUTPUT_DIR / 'best_d3_agentic_system.generations.csv', index=True)

submission = submission_frame(final_generations['Predicted'])
submission.to_csv(SUBMISSION_FILENAME, index=True)
with open(OUTPUT_DIR / 'best_d3_agentic_system.json', 'w', encoding='utf-8') as file:
    json.dump({'model_key': MODEL_KEY, 'model': MODEL_NAME, 'strategy': best_strategy, 'selection_metric': SELECTION_METRIC, 'selection_tie_breaker': SELECTION_TIE_BREAKER}, file, indent=2)

print(f'Saved Kaggle submission: {SUBMISSION_FILENAME}')
print(f'Rows: {len(submission)}')
print('Prediction distribution:')
display(submission['Predicted'].value_counts().rename_axis('Predicted').reset_index(name='count'))
display(submission.head(10))

## Final Notes

The report should discuss whether D3 improves over the same-scale single-agent comparator, and how both compare to HW1 vector systems, HW2 encoder-only fine-tuning, and HW3 single-invocation prompting. The key qualitative evidence is in the D3 intermediate columns: `question_intent_generation`, `answer_content_generation`, and `gap_evasion_generation`.